# TechTrove E-Commerce — Data Integration Pipeline

**รายวิชา:** Data Warehousing / Data Engineering — Week 8 Lab
**บทบาท:** Data Engineer
**เป้าหมาย:** รวมข้อมูลคำสั่งซื้อ (CSV), ลูกค้า (CSV), สินค้า (Excel) และการชำระเงิน (JSON) ที่กระจายอยู่คนละระบบ
ให้เป็น Fact/Dimension table ที่พร้อมสำหรับการวิเคราะห์ยอดขายสุทธิ พร้อมจัดทำ Data Quality Report ที่ตรวจสอบย้อนกลับได้

**โครงสร้าง Notebook**
1. Extract & Profile ข้อมูลดิบทั้ง 5 ไฟล์
2. Schema Alignment & Combine orders (ม.ค. + ก.พ.)
3. Transform: แปลงชนิดข้อมูล ทำความสะอาด ทำมาตรฐาน และลบข้อมูลซ้ำ
4. Integrate & Validate: merge กับ customer/product/payment master พร้อมตรวจ cardinality/unmatched
5. Load: สร้าง `dim_customer.csv`, `dim_product.csv`, `fact_sales.csv`, `data_quality_report.csv`
6. Analyze: `summary_by_province.csv`, `summary_by_category.csv`
7. Challenge: `validate_data()` + Data-Quality Funnel
8. สรุปคุณภาพข้อมูลก่อน/หลัง Integration และคำตอบคำถามวิเคราะห์ 6 ข้อ

> **กติกาทางธุรกิจที่ยึดตามตลอด pipeline นี้:**
> 1. เก็บ `order_id` เพียงหนึ่งแถว หากซ้ำให้เก็บแถวล่าสุดตามลำดับที่ปรากฏ
> 2. `quantity > 0`, `unit_price > 0`, `0 <= discount <= 1`
> 3. ลูกค้าและสินค้าต้องพบใน Master Data
> 4. นับเป็นยอดขายเมื่อ `payment.status == PAID` เท่านั้น
> 5. `net_sales = quantity × unit_price × (1 - discount)`
> 6. ห้ามลบข้อมูลผิดปกติโดยไม่บันทึกจำนวนและเหตุผลใน Data Quality Report


In [7]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

BASE_DIR = Path.cwd()
DATA = BASE_DIR # Corrected: files are directly in BASE_DIR, not in a 'data' subfolder
OUTPUT = BASE_DIR / "output"
OUTPUT.mkdir(exist_ok=True)

# เก็บสถิติสำหรับ Data Quality Report แบบตรวจสอบย้อนกลับได้
dq_log = []  # each item: dict(stage, issue, rows_affected, action)


def log_dq(stage, issue, rows_affected, action):
    dq_log.append(
        {
            "stage": stage,
            "issue": issue,
            "rows_affected": int(rows_affected),
            "action": action,
        }
    )


def profile_df(name, df):
    """แสดง shape, dtype, missing, duplicate ของ dataframe ก่อนแก้ไข"""
    print(f"\n--- Profile: {name} ---")
    print("shape:", df.shape)
    print("dtypes:\n", df.dtypes)
    print("missing values:\n", df.isna().sum())
    print("duplicate rows (all columns):", df.duplicated().sum())
    print("sample:\n", df.head(3))

## 5.1 Extract & Profile
อ่านข้อมูลจากทั้ง 3 ชนิดไฟล์ (CSV / Excel / JSON) และสำรวจคุณภาพเบื้องต้น (shape, dtype, missing, duplicate) ก่อนแก้ไขใดๆ

In [8]:
print("=" * 70)
print("STEP 5.1  EXTRACT & PROFILE")
print("=" * 70)

orders_jan_raw = pd.read_csv(DATA / "orders_2026_01.csv")
orders_feb_raw = pd.read_csv(DATA / "orders_2026_02.csv")
customers_raw = pd.read_csv(DATA / "customers_crm.csv")
products_raw = pd.read_excel(DATA / "product_master.xlsx")

with open(DATA / "payments.json", encoding="utf-8") as f:
    payments_json = json.load(f)
payments_raw = pd.json_normalize(payments_json)

profile_df("orders_2026_01.csv", orders_jan_raw)
profile_df("orders_2026_02.csv", orders_feb_raw)
profile_df("customers_crm.csv", customers_raw)
profile_df("product_master.xlsx", products_raw)
profile_df("payments.json (normalized)", payments_raw)

# สรุปคุณภาพข้อมูล "ก่อน" การ integrate (raw) เพื่อเปรียบเทียบกับ "หลัง"
raw_quality_summary = pd.DataFrame(
    [
        {
            "dataset": "orders_2026_01",
            "rows": len(orders_jan_raw),
            "duplicate_rows": orders_jan_raw.duplicated().sum(),
            "duplicate_order_id": orders_jan_raw["order_id"].duplicated().sum(),
            "missing_values_total": int(orders_jan_raw.isna().sum().sum()),
        },
        {
            "dataset": "orders_2026_02",
            "rows": len(orders_feb_raw),
            "duplicate_rows": orders_feb_raw.duplicated().sum(),
            "duplicate_order_id": orders_feb_raw["order_id"].duplicated().sum(),
            "missing_values_total": int(orders_feb_raw.isna().sum().sum()),
        },
        {
            "dataset": "customers_crm",
            "rows": len(customers_raw),
            "duplicate_rows": customers_raw.duplicated().sum(),
            "duplicate_order_id": customers_raw["customer_id"].duplicated().sum(),
            "missing_values_total": int(customers_raw.isna().sum().sum()),
        },
        {
            "dataset": "product_master",
            "rows": len(products_raw),
            "duplicate_rows": products_raw.duplicated().sum(),
            "duplicate_order_id": products_raw["product_id"].duplicated().sum(),
            "missing_values_total": int(products_raw.isna().sum().sum()),
        },
        {
            "dataset": "payments",
            "rows": len(payments_raw),
            "duplicate_rows": payments_raw.duplicated().sum(),
            "duplicate_order_id": payments_raw["order_id"].duplicated().sum(),
            "missing_values_total": int(payments_raw.isna().sum().sum()),
        },
    ]
)
print("\n=== สรุปคุณภาพข้อมูลก่อน Integration (RAW) ===")
print(raw_quality_summary.to_string(index=False))
raw_quality_summary.to_csv(OUTPUT / "_before_quality_summary.csv", index=False)

STEP 5.1  EXTRACT & PROFILE

--- Profile: orders_2026_01.csv ---
shape: (361, 8)
dtypes:
 order_id        object
order_date      object
customer_id     object
product_id      object
quantity         int64
unit_price     float64
discount       float64
channel         object
dtype: object
missing values:
 order_id       0
order_date     0
customer_id    0
product_id     0
quantity       0
unit_price     1
discount       0
channel        0
dtype: int64
duplicate rows (all columns): 1
sample:
     order_id           order_date customer_id product_id  quantity  unit_price  discount      channel
0  ORD000001  2026-01-08 17:11:00       C0158       P039         2       940.5      0.00  Marketplace
1  ORD000002  2026-01-12 20:13:00       C0123       P038         1     12900.0      0.10   Mobile App
2  ORD000003  2026-01-26 14:00:00       C0119       P031         3      1415.5      0.05          Web

--- Profile: orders_2026_02.csv ---
shape: (391, 8)
dtypes:
 order_id         object
ordered_at 

## 5.2 Schema Alignment & Combine Orders
ไฟล์ orders ของเดือนกุมภาพันธ์มีชื่อคอลัมน์ รูปแบบวันที่ และรูปแบบส่วนลดต่างจากเดือนมกราคม (schema drift) จึงต้องปรับให้ตรงกันก่อน แล้วจึง `pd.concat(..., ignore_index=True)`

In [9]:
print("\n" + "=" * 70)
print("STEP 5.2  SCHEMA ALIGNMENT & COMBINE ORDERS")
print("=" * 70)

n_jan_raw, n_feb_raw = len(orders_jan_raw), len(orders_feb_raw)

# --- January: schema เป็น "มาตรฐาน" ที่จะยึดตาม ---
#   order_id, order_date (ISO), customer_id, product_id, quantity,
#   unit_price, discount (0-1 float), channel

# --- February: ปรับให้ตรงกับ schema ของมกราคม ---
#   ordered_at        -> order_date   (dd/mm/yyyy HH:MM  -> ISO)
#   qty               -> quantity
#   discount_pct (str "5%") -> discount (float 0-1)
orders_feb_aligned = orders_feb_raw.rename(
    columns={
        "ordered_at": "order_date",
        "qty": "quantity",
    }
).copy()

# แปลงรูปแบบวันที่ dd/mm/yyyy HH:MM -> เดียวกับมกราคม (datetime)
orders_feb_aligned["order_date"] = pd.to_datetime(
    orders_feb_aligned["order_date"], format="%d/%m/%Y %H:%M", errors="coerce"
)

# แปลง "5%" -> 0.05
orders_feb_aligned["discount"] = (
    orders_feb_aligned["discount_pct"]
    .astype(str)
    .str.strip()
    .str.replace("%", "", regex=False)
    .astype(float)
    / 100.0
)
orders_feb_aligned = orders_feb_aligned.drop(columns=["discount_pct"])

# จัดลำดับคอลัมน์ให้ตรงกับ orders_jan
STANDARD_ORDER_COLS = [
    "order_id",
    "order_date",
    "customer_id",
    "product_id",
    "quantity",
    "unit_price",
    "discount",
    "channel",
]
orders_jan_aligned = orders_jan_raw.copy()
orders_jan_aligned["order_date"] = pd.to_datetime(
    orders_jan_aligned["order_date"], errors="coerce"
)
orders_jan_aligned = orders_jan_aligned[STANDARD_ORDER_COLS]
orders_feb_aligned = orders_feb_aligned[STANDARD_ORDER_COLS]

print("Schema drift ที่พบและแก้ไข:")
print(" - ordered_at (dd/mm/yyyy HH:MM, ก.พ.)  -> order_date (ISO datetime, เหมือน ม.ค.)")
print(" - qty (ก.พ.)                            -> quantity")
print(" - discount_pct ('5%', string, ก.พ.)     -> discount (float 0-1, เหมือน ม.ค.)")

orders_combined = pd.concat(
    [orders_jan_aligned, orders_feb_aligned], ignore_index=True
)
n_combined_raw = len(orders_combined)
print(f"\nแถวหลัง concat (ก่อนลบ duplicate): {n_combined_raw} "
      f"(ม.ค. {n_jan_raw} + ก.พ. {n_feb_raw})")

log_dq(
    "5.2 combine",
    "รวม orders ม.ค.+ก.พ. หลัง schema alignment",
    n_combined_raw,
    "concat(ignore_index=True) สำเร็จ",
)


STEP 5.2  SCHEMA ALIGNMENT & COMBINE ORDERS
Schema drift ที่พบและแก้ไข:
 - ordered_at (dd/mm/yyyy HH:MM, ก.พ.)  -> order_date (ISO datetime, เหมือน ม.ค.)
 - qty (ก.พ.)                            -> quantity
 - discount_pct ('5%', string, ก.พ.)     -> discount (float 0-1, เหมือน ม.ค.)

แถวหลัง concat (ก่อนลบ duplicate): 752 (ม.ค. 361 + ก.พ. 391)


## 5.3 Transform: แปลงชนิดข้อมูล / ทำความสะอาด / ทำมาตรฐาน / ลบข้อมูลซ้ำ (orders)
แปลงชนิดข้อมูล, จัดการค่าว่างของ `unit_price`, ลบ `order_id` ที่ซ้ำ (เก็บแถวล่าสุด) และตรวจกติกาทางธุรกิจ (`quantity>0`, `unit_price>0`, `0<=discount<=1`) ทุกการคัดออกถูกบันทึกไว้ใน `dq_log`

In [10]:
print("\n" + "=" * 70)
print("STEP 5.3  TRANSFORM: TYPES / CLEANING / STANDARDIZATION / DEDUP")
print("=" * 70)

orders = orders_combined.copy()

# ---- แปลงชนิดข้อมูลให้ถูกต้อง ----
orders["order_id"] = orders["order_id"].astype(str).str.strip()
orders["customer_id"] = orders["customer_id"].astype(str).str.strip()
orders["product_id"] = orders["product_id"].astype(str).str.strip()
orders["channel"] = orders["channel"].astype(str).str.strip()
orders["quantity"] = pd.to_numeric(orders["quantity"], errors="coerce")
orders["unit_price"] = pd.to_numeric(orders["unit_price"], errors="coerce")
orders["discount"] = pd.to_numeric(orders["discount"], errors="coerce")

# ---- ค่าว่างของ unit_price ----
missing_price = orders["unit_price"].isna().sum()
if missing_price:
    log_dq(
        "5.3 cleaning",
        "unit_price เป็นค่าว่าง",
        missing_price,
        "คัดออก (ไม่สามารถคำนวณ net_sales ได้โดยไม่เดาราคา)",
    )
orders = orders.dropna(subset=["unit_price"])

# ---- แถวซ้ำตาม order_id: กติกา -> เก็บแถวล่าสุดตามลำดับที่ปรากฏ ----
dup_order_id_mask = orders.duplicated(subset=["order_id"], keep="last")
n_dup_order_id = dup_order_id_mask.sum()
if n_dup_order_id:
    log_dq(
        "5.3 cleaning",
        "order_id ซ้ำ (มากกว่า 1 แถวต่อคำสั่งซื้อ)",
        n_dup_order_id,
        "เก็บเฉพาะแถวล่าสุดตามลำดับที่ปรากฏ (keep='last') ตามกติกาข้อ 1",
    )
orders = orders.drop_duplicates(subset=["order_id"], keep="last")

n_after_dedup = len(orders)
print(f"แถวหลัง concat: {n_combined_raw}  ->  แถวหลังลบ duplicate order_id: {n_after_dedup}")

# ---- Business rule validation: quantity>0, unit_price>0, 0<=discount<=1 ----
bad_qty = orders["quantity"] <= 0
bad_price = orders["unit_price"] <= 0
bad_discount = (orders["discount"] < 0) | (orders["discount"] > 1)

if bad_qty.sum():
    log_dq(
        "5.3 business rule",
        "quantity <= 0 (ค่าผิดปกติ เช่น -1)",
        bad_qty.sum(),
        "คัดออกจาก fact table ตามกติกาข้อ 2",
    )
if bad_price.sum():
    log_dq(
        "5.3 business rule",
        "unit_price <= 0",
        bad_price.sum(),
        "คัดออกจาก fact table ตามกติกาข้อ 2",
    )
if bad_discount.sum():
    log_dq(
        "5.3 business rule",
        "discount อยู่นอกช่วง [0,1]",
        bad_discount.sum(),
        "คัดออกจาก fact table ตามกติกาข้อ 2",
    )

orders_valid = orders[~(bad_qty | bad_price | bad_discount)].copy()
print(f"แถวหลังตรวจกติกาทางธุรกิจ (quantity/unit_price/discount): {len(orders_valid)}")


STEP 5.3  TRANSFORM: TYPES / CLEANING / STANDARDIZATION / DEDUP
แถวหลัง concat: 752  ->  แถวหลังลบ duplicate order_id: 748
แถวหลังตรวจกติกาทางธุรกิจ (quantity/unit_price/discount): 746


### ทำความสะอาด `customers_crm`
ปรับอีเมลเป็นตัวพิมพ์เล็ก/ตัดช่องว่าง, ทำชื่อจังหวัดให้เป็นมาตรฐานเดียว (เช่น `Chonburi`/`ชลบุรี ` → `ชลบุรี`, `กทม.`/`Bangkok` → `กรุงเทพมหานคร`) และลบแถวลูกค้าที่ซ้ำ

In [11]:
customers = customers_raw.copy()
customers["customer_id"] = customers["customer_id"].astype(str).str.strip()
customers["full_name"] = customers["full_name"].astype(str).str.strip()

# email -> lower-case + trim
missing_email = customers["email"].isna().sum()
if missing_email:
    log_dq(
        "5.3 customers cleaning",
        "email เป็นค่าว่าง",
        missing_email,
        "คงแถวไว้ (ไม่กระทบ key การ merge) แต่บันทึกไว้ใน DQ report",
    )
customers["email"] = customers["email"].astype(str).str.strip().str.lower()
customers.loc[customers["email"].isin(["nan", "none", ""]), "email"] = np.nan

# province -> ปรับให้เป็นชื่อมาตรฐานเดียว (ภาษาไทยทางการ)
PROVINCE_MAP = {
    "chonburi": "ชลบุรี",
    "ชลบุรี": "ชลบุรี",
    "ชลบุรี ": "ชลบุรี",
    "khon kaen": "ขอนแก่น",
    "ขอนแก่น": "ขอนแก่น",
    "ขอนเเก่น": "ขอนแก่น",  # สระ เ ซ้ำ (พิมพ์ผิด)
    "bangkok": "กรุงเทพมหานคร",
    "กรุงเทพมหานคร": "กรุงเทพมหานคร",
    "กทม.": "กรุงเทพมหานคร",
    "กทม": "กรุงเทพมหานคร",
    "rayong": "ระยอง",
    "ระยอง": "ระยอง",
    "phuket": "ภูเก็ต",
    "ภูเก็ต": "ภูเก็ต",
    "chiang mai": "เชียงใหม่",
    "เชียงใหม่": "เชียงใหม่",
}


def standardize_province(x):
    if pd.isna(x):
        return x
    key = str(x).strip().lower()
    return PROVINCE_MAP.get(key, str(x).strip())


before_unique_provinces = customers_raw["province"].nunique()
customers["province"] = customers["province"].apply(standardize_province)
after_unique_provinces = customers["province"].nunique()
print(
    f"\nจำนวนชื่อจังหวัด (unique) ก่อนทำมาตรฐาน: {before_unique_provinces} "
    f"-> หลังทำมาตรฐาน: {after_unique_provinces}"
)
log_dq(
    "5.3 customers cleaning",
    f"ชื่อจังหวัดสะกด/ภาษาไม่ตรงกัน ({before_unique_provinces} รูปแบบ)",
    len(customers),
    f"map ให้เป็นชื่อมาตรฐานเดียว เหลือ {after_unique_provinces} จังหวัด",
)

customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")

# dedup ลูกค้า: เก็บ 1 แถวต่อ customer_id (ข้อมูลเหมือนกันทุกคอลัมน์ -> เก็บแถวแรก)
n_cust_before = len(customers)
n_dup_cust = customers.duplicated(subset=["customer_id"]).sum()
if n_dup_cust:
    log_dq(
        "5.3 customers cleaning",
        "customer_id ซ้ำใน customers_crm",
        n_dup_cust,
        "เก็บแถวแรกต่อ customer_id (keep='first')",
    )
customers = customers.drop_duplicates(subset=["customer_id"], keep="first")
print(f"customers: {n_cust_before} -> {len(customers)} หลัง dedup")


จำนวนชื่อจังหวัด (unique) ก่อนทำมาตรฐาน: 14 -> หลังทำมาตรฐาน: 6
customers: 163 -> 160 หลัง dedup


### ทำความสะอาด `product_master`
แปลงชนิดข้อมูลและปรับ `active_flag` ให้เป็นมาตรฐาน (Y/N)

In [12]:
products = products_raw.copy()
products["product_id"] = products["product_id"].astype(str).str.strip()
products["category"] = products["category"].astype(str).str.strip()
products["active_flag"] = products["active_flag"].astype(str).str.strip().str.upper()
products["standard_price"] = pd.to_numeric(products["standard_price"], errors="coerce")

n_inactive = (products["active_flag"] == "N").sum()
log_dq(
    "5.3 products info",
    "สินค้าที่ active_flag = N (เลิกขาย)",
    n_inactive,
    "คงไว้ใน dim_product (เพื่อ reference คำสั่งซื้อในอดีต) แต่ทำเครื่องหมายไว้",
)

### ทำความสะอาด `payments`
ลบ payment event ที่ซ้ำทุกคอลัมน์ และเก็บ payment ล่าสุดต่อ `order_id` กรณีมีมากกว่าหนึ่งเหตุการณ์

In [13]:
payments = payments_raw.rename(
    columns={"payment.method": "payment_method", "payment.status": "payment_status"}
).copy()
payments["order_id"] = payments["order_id"].astype(str).str.strip()
payments["payment_status"] = payments["payment_status"].astype(str).str.strip().str.upper()
payments["paid_at"] = pd.to_datetime(payments["paid_at"], errors="coerce")

n_pay_dup = payments.duplicated().sum()
if n_pay_dup:
    log_dq(
        "5.3 payments cleaning",
        "payment event ซ้ำทุกคอลัมน์",
        n_pay_dup,
        "ลบซ้ำ (drop_duplicates ทั้งแถว)",
    )
payments = payments.drop_duplicates()

# เก็บ payment ล่าสุดต่อ order_id (ถ้า order เดียวมีหลาย payment event ที่ต่างกัน)
n_pay_multi_order = payments.duplicated(subset=["order_id"], keep=False).sum()
if payments.duplicated(subset=["order_id"]).sum():
    log_dq(
        "5.3 payments cleaning",
        "order_id เดียวกันมี payment event มากกว่า 1 ครั้ง (คนละเนื้อหา)",
        payments.duplicated(subset=["order_id"]).sum(),
        "เก็บ payment ล่าสุดตาม paid_at ต่อ order_id",
    )
payments = payments.sort_values("paid_at").drop_duplicates(subset=["order_id"], keep="last")

print(f"\npayments: {len(payments_raw)} (raw) -> {len(payments)} หลังลบ duplicate")
print("payment_status counts:\n", payments["payment_status"].value_counts())


payments: 752 (raw) -> 751 หลังลบ duplicate
payment_status counts:
 payment_status
PAID        687
FAILED       46
REFUNDED     18
Name: count, dtype: int64


## 5.4 Integrate & Validate
เชื่อมโยง orders กับ customer, product และ payment ด้วย `pd.merge()` โดยระบุ `validate=` (ตรวจ cardinality) และ `indicator=True` (ตรวจแถวที่จับคู่ไม่ได้)

In [14]:
print("\n" + "=" * 70)
print("STEP 5.4  INTEGRATE (merge) & VALIDATE")
print("=" * 70)

valid_customer_ids = set(customers["customer_id"])
valid_product_ids = set(products["product_id"])

unmatched_customer_mask = ~orders_valid["customer_id"].isin(valid_customer_ids)
unmatched_product_mask = ~orders_valid["product_id"].isin(valid_product_ids)

n_unmatched_customer = unmatched_customer_mask.sum()
n_unmatched_product = unmatched_product_mask.sum()

print(f"orders ที่ customer_id ไม่พบใน Master Data: {n_unmatched_customer}")
print(f"orders ที่ product_id ไม่พบใน Master Data: {n_unmatched_product}")

if n_unmatched_customer:
    log_dq(
        "5.4 integration",
        "customer_id ไม่พบใน customers_crm (master)",
        n_unmatched_customer,
        "คัดออกจาก fact_sales ตามกติกาข้อ 3 (ลูกค้าต้องพบใน Master Data)",
    )
if n_unmatched_product:
    log_dq(
        "5.4 integration",
        "product_id ไม่พบใน product_master (master)",
        n_unmatched_product,
        "คัดออกจาก fact_sales ตามกติกาข้อ 3 (สินค้าต้องพบใน Master Data)",
    )

# merge ลูกค้า (many orders -> 1 customer : many_to_one)
orders_cust = pd.merge(
    orders_valid,
    customers[["customer_id", "full_name", "email", "province"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator="_merge_customer",
)

# merge สินค้า (many orders -> 1 product : many_to_one)
orders_cust_prod = pd.merge(
    orders_cust,
    products[["product_id", "product_name", "category", "standard_price", "active_flag"]],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="_merge_product",
)

# merge การชำระเงิน (1 order -> 1 payment event : one_to_one)
orders_full = pd.merge(
    orders_cust_prod,
    payments[["order_id", "payment_method", "payment_status", "paid_at"]],
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator="_merge_payment",
)

print("\nสรุปการ merge (indicator):")
print(" customer:", orders_full["_merge_customer"].value_counts().to_dict())
print(" product :", orders_full["_merge_product"].value_counts().to_dict())
print(" payment :", orders_full["_merge_payment"].value_counts().to_dict())

n_payment_unmatched = (orders_full["_merge_payment"] == "left_only").sum()
if n_payment_unmatched:
    log_dq(
        "5.4 integration",
        "order_id ไม่พบ payment event ที่สอดคล้อง (orphan order)",
        n_payment_unmatched,
        "ไม่นับเป็นยอดขาย (ไม่มี payment.status = PAID) ตามกติกาข้อ 4",
    )

# ตรวจ orphan payments: payment ที่ order_id ไม่พบใน orders_valid เลย
orphan_payments_mask = ~payments["order_id"].isin(orders_valid["order_id"])
n_orphan_payments = orphan_payments_mask.sum()
if n_orphan_payments:
    log_dq(
        "5.4 integration",
        "payment event ที่ order_id ไม่พบใน orders ที่ผ่านการตรวจสอบ (orphan payment)",
        n_orphan_payments,
        "ไม่ถูกใช้ในการคำนวณยอดขาย (ไม่มีคำสั่งซื้อรองรับ)",
    )


STEP 5.4  INTEGRATE (merge) & VALIDATE
orders ที่ customer_id ไม่พบใน Master Data: 22
orders ที่ product_id ไม่พบใน Master Data: 2

สรุปการ merge (indicator):
 customer: {'both': 724, 'left_only': 22, 'right_only': 0}
 product : {'both': 744, 'left_only': 2, 'right_only': 0}
 payment : {'both': 746, 'left_only': 0, 'right_only': 0}


### คำนวณ net_sales และคัดเฉพาะธุรกรรมที่ใช้ได้จริง
`net_sales = quantity × unit_price × (1 - discount)` และนับเป็นยอดขายเฉพาะเมื่อพบทั้งลูกค้า/สินค้าใน Master Data **และ** `payment_status == PAID`

In [15]:
orders_full["net_sales"] = (
    orders_full["quantity"] * orders_full["unit_price"] * (1 - orders_full["discount"])
)

n_not_paid = (orders_full["payment_status"] != "PAID").sum()
log_dq(
    "5.4 business rule",
    "payment_status ไม่ใช่ PAID (FAILED / REFUNDED / ไม่มี payment)",
    n_not_paid,
    "ไม่นับเป็นยอดขายสุทธิ ตามกติกาข้อ 4",
)

# กติกาข้อ 3: ต้องพบทั้งลูกค้าและสินค้าใน Master Data (ไม่ใช่แค่ merge แบบ left
# แล้วปล่อยให้ NaN หลุดเข้า fact table) + กติกาข้อ 4: payment_status ต้องเป็น PAID
is_matched_customer = orders_full["_merge_customer"] == "both"
is_matched_product = orders_full["_merge_product"] == "both"
is_paid = orders_full["payment_status"] == "PAID"

fact_sales = orders_full[is_matched_customer & is_matched_product & is_paid].copy()

print(f"\nจำนวนธุรกรรมยอดขายที่ใช้ได้จริง (PAID): {len(fact_sales)}")
print(f"ยอดขายสุทธิรวม: {fact_sales['net_sales'].sum():,.2f} บาท")


จำนวนธุรกรรมยอดขายที่ใช้ได้จริง (PAID): 660
ยอดขายสุทธิรวม: 10,224,044.08 บาท


## 5.5 Load
เขียนไฟล์ผลลัพธ์ 4 ไฟล์: `dim_customer.csv`, `dim_product.csv`, `fact_sales.csv`, `data_quality_report.csv`

In [16]:
print("\n" + "=" * 70)
print("STEP 5.5  LOAD OUTPUT FILES")
print("=" * 70)

dim_customer = customers[["customer_id", "full_name", "email", "province", "signup_date"]].copy()
dim_customer.to_csv(OUTPUT / "dim_customer.csv", index=False)

dim_product = products[
    ["product_id", "product_name", "category", "standard_price", "active_flag"]
].copy()
dim_product.to_csv(OUTPUT / "dim_product.csv", index=False)

fact_sales_out = fact_sales[
    [
        "order_id",
        "order_date",
        "customer_id",
        "product_id",
        "quantity",
        "unit_price",
        "discount",
        "channel",
        "province",
        "category",
        "payment_method",
        "payment_status",
        "paid_at",
        "net_sales",
    ]
].copy()
fact_sales_out.to_csv(OUTPUT / "fact_sales.csv", index=False)

data_quality_report = pd.DataFrame(dq_log)
data_quality_report.to_csv(OUTPUT / "data_quality_report.csv", index=False)

print("เขียนไฟล์: dim_customer.csv, dim_product.csv, fact_sales.csv, data_quality_report.csv")


STEP 5.5  LOAD OUTPUT FILES
เขียนไฟล์: dim_customer.csv, dim_product.csv, fact_sales.csv, data_quality_report.csv


## 5.6 Analyze
สรุปยอดขายสุทธิแยกตามจังหวัดและหมวดสินค้า

In [17]:
print("\n" + "=" * 70)
print("STEP 5.6  ANALYZE")
print("=" * 70)

summary_by_province = (
    fact_sales_out.groupby("province", as_index=False)
    .agg(transactions=("order_id", "count"), net_sales=("net_sales", "sum"))
    .sort_values("net_sales", ascending=False)
)
summary_by_province.to_csv(OUTPUT / "summary_by_province.csv", index=False)

summary_by_category = (
    fact_sales_out.groupby("category", as_index=False)
    .agg(transactions=("order_id", "count"), net_sales=("net_sales", "sum"))
    .sort_values("net_sales", ascending=False)
)
summary_by_category.to_csv(OUTPUT / "summary_by_category.csv", index=False)

print("\n=== summary_by_province ===")
print(summary_by_province.to_string(index=False))
print("\n=== summary_by_category ===")
print(summary_by_category.to_string(index=False))


STEP 5.6  ANALYZE

=== summary_by_province ===
     province  transactions    net_sales
กรุงเทพมหานคร           154 2612955.8700
      ขอนแก่น           110 2031943.3850
        ระยอง           120 1523168.6150
    เชียงใหม่           104 1477338.0075
       ภูเก็ต            86 1427388.7350
       ชลบุรี            86 1151249.4650

=== summary_by_category ===
  category  transactions    net_sales
Smartphone           178 3092117.3425
 Accessory           180 2710582.7800
  Notebook           161 2221495.4875
Smart Home           141 2199848.4675


### สรุปคุณภาพข้อมูล "หลัง" Integration และ Data-Quality Funnel

In [18]:
after_quality_summary = pd.DataFrame(
    [
        {"metric": "orders รวมหลัง concat (ก่อนลบ duplicate)", "value": n_combined_raw},
        {"metric": "orders หลังลบ duplicate order_id", "value": n_after_dedup},
        {"metric": "orders ผ่านกติกาทางธุรกิจ (qty/price/discount)", "value": len(orders_valid)},
        {"metric": "orders ที่ customer_id ไม่พบใน master", "value": int(n_unmatched_customer)},
        {"metric": "orders ที่ product_id ไม่พบใน master", "value": int(n_unmatched_product)},
        {"metric": "payment events ไม่ใช่ PAID", "value": int(n_not_paid)},
        {"metric": "ธุรกรรมยอดขายสุทธิที่ใช้ได้จริง (PAID)", "value": len(fact_sales)},
        {"metric": "ยอดขายสุทธิรวม (บาท)", "value": round(float(fact_sales["net_sales"].sum()), 2)},
        {"metric": "จำนวนจังหวัดมาตรฐาน (unique)", "value": after_unique_provinces},
    ]
)
after_quality_summary.to_csv(OUTPUT / "_after_quality_summary.csv", index=False)
print("\n=== สรุปคุณภาพข้อมูลหลัง Integration (AFTER) ===")
print(after_quality_summary.to_string(index=False))

print("\n" + "=" * 70)
print("STEP: DATA-QUALITY FUNNEL (raw -> deduplicated -> matched -> paid sales)")
print("=" * 70)
funnel = pd.DataFrame(
    [
        {"stage": "1. Raw (concat)", "rows": n_combined_raw},
        {"stage": "2. Deduplicated", "rows": n_after_dedup},
        {"stage": "3. Business-rule valid", "rows": len(orders_valid)},
        {
            "stage": "4. Matched to Master (customer & product)",
            "rows": int((is_matched_customer & is_matched_product).sum()),
        },
        {"stage": "5. Paid sales (fact_sales)", "rows": len(fact_sales)},
    ]
)
print(funnel.to_string(index=False))
funnel.to_csv(OUTPUT / "_data_quality_funnel.csv", index=False)


=== สรุปคุณภาพข้อมูลหลัง Integration (AFTER) ===
                                        metric       value
      orders รวมหลัง concat (ก่อนลบ duplicate)      752.00
              orders หลังลบ duplicate order_id      748.00
orders ผ่านกติกาทางธุรกิจ (qty/price/discount)      746.00
         orders ที่ customer_id ไม่พบใน master       22.00
          orders ที่ product_id ไม่พบใน master        2.00
                    payment events ไม่ใช่ PAID       63.00
        ธุรกรรมยอดขายสุทธิที่ใช้ได้จริง (PAID)      660.00
                          ยอดขายสุทธิรวม (บาท) 10224044.08
                  จำนวนจังหวัดมาตรฐาน (unique)        6.00

STEP: DATA-QUALITY FUNNEL (raw -> deduplicated -> matched -> paid sales)
                                    stage  rows
                          1. Raw (concat)   752
                          2. Deduplicated   748
                   3. Business-rule valid   746
4. Matched to Master (customer & product)   722
               5. Paid sales (fact_sales)   66

## Challenge (+2 คะแนน): `validate_data()` และ Data-Quality Funnel
ฟังก์ชันตรวจสอบ uniqueness, referential integrity และค่านอกช่วงของ `fact_sales` ก่อนนำไปใช้งานจริง พร้อมกราฟ funnel แสดงจำนวนแถวที่ลดลงในแต่ละขั้นตอน (raw → deduplicated → business-rule valid → matched to master → paid sales)

In [19]:
print("\n" + "=" * 70)
print("CHALLENGE: validate_data() and funnel chart")
print("=" * 70)


def validate_data(fact_df, dim_customer_df, dim_product_df):
    """
    ตรวจสอบคุณภาพของ fact_sales ก่อนนำไปใช้งานจริง:
      1) uniqueness ของ order_id ใน fact table
      2) referential integrity: customer_id / product_id ต้องพบใน dimension
      3) ค่าที่อยู่นอกช่วงที่ยอมรับได้ (quantity>0, unit_price>0, 0<=discount<=1)
      4) payment_status ต้องเป็น PAID เท่านั้น
    หากพบปัญหา จะ raise AssertionError พร้อมข้อความอธิบาย
    """
    errors = []

    if fact_df["order_id"].duplicated().any():
        errors.append(
            f"order_id ซ้ำใน fact_sales จำนวน {fact_df['order_id'].duplicated().sum()} แถว"
        )

    unknown_cust = ~fact_df["customer_id"].isin(dim_customer_df["customer_id"])
    if unknown_cust.any():
        errors.append(f"customer_id ไม่พบใน dim_customer จำนวน {unknown_cust.sum()} แถว")

    unknown_prod = ~fact_df["product_id"].isin(dim_product_df["product_id"])
    if unknown_prod.any():
        errors.append(f"product_id ไม่พบใน dim_product จำนวน {unknown_prod.sum()} แถว")

    if (fact_df["quantity"] <= 0).any():
        errors.append("พบ quantity <= 0 ใน fact_sales")

    if (fact_df["unit_price"] <= 0).any():
        errors.append("พบ unit_price <= 0 ใน fact_sales")

    if ((fact_df["discount"] < 0) | (fact_df["discount"] > 1)).any():
        errors.append("พบ discount นอกช่วง [0, 1] ใน fact_sales")

    if (fact_df["payment_status"] != "PAID").any():
        errors.append("พบแถวที่ payment_status ไม่ใช่ PAID ปนอยู่ใน fact_sales")

    if errors:
        raise AssertionError("validate_data พบปัญหา:\n- " + "\n- ".join(errors))

    print("validate_data: PASSED — fact_sales ผ่านการตรวจสอบทุกกติกา ✅")
    return True


validate_data(fact_sales_out, dim_customer, dim_product)

# กราฟ data-quality funnel
try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 5))
    # ใช้ label ภาษาอังกฤษเพื่อหลีกเลี่ยงปัญหา font ไม่รองรับตัวอักษรไทยในกราฟ
    stage_labels_en = [
        "1. Raw (concat)",
        "2. Deduplicated",
        "3. Business-rule valid",
        "4. Matched to Master",
        "5. Paid sales (fact_sales)",
    ]
    values = funnel["rows"].tolist()
    bars = ax.barh(stage_labels_en, values, color="#4C72B0")
    ax.invert_yaxis()
    ax.set_xlabel("Row count")
    ax.set_title("Data Quality Funnel: Raw -> Deduplicated -> Matched -> Paid Sales")
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2, f"{val:,}",
                 va="center")
    plt.tight_layout()
    fig.savefig(OUTPUT / "_data_quality_funnel.png", dpi=150)
    plt.close(fig)
    print("บันทึกกราฟ: output/_data_quality_funnel.png")
except Exception as e:
    print("ไม่สามารถวาดกราฟได้:", e)

print("\nเสร็จสิ้น Data Integration Pipeline — ดูผลลัพธ์ทั้งหมดในโฟลเดอร์ output/")


CHALLENGE: validate_data() and funnel chart
validate_data: PASSED — fact_sales ผ่านการตรวจสอบทุกกติกา ✅
บันทึกกราฟ: output/_data_quality_funnel.png

เสร็จสิ้น Data Integration Pipeline — ดูผลลัพธ์ทั้งหมดในโฟลเดอร์ output/



## 8. สรุปคุณภาพข้อมูลก่อน/หลัง Integration และคำตอบคำถามวิเคราะห์ 6 ข้อ

### สรุปคุณภาพข้อมูลก่อน Integration (RAW)

| dataset | rows | duplicate rows | duplicate key | missing values |
|---|---|---|---|---|
| orders_2026_01 | 361 | 1 | 1 (order_id) | 1 (unit_price) |
| orders_2026_02 | 391 | 1 | 1 (order_id) | 1 (unit_price) |
| customers_crm | 163 | 0 | 3 (customer_id) | 5 (email) |
| product_master | 40 | 0 | 0 | 0 |
| payments | 752 | 1 | 1 (order_id) | 0 |

ปัญหาหลักที่พบตอน profiling: **schema ไม่ตรงกัน** ระหว่าง orders สองเดือน (ชื่อคอลัมน์, รูปแบบวันที่, รูปแบบส่วนลด), **ชื่อจังหวัดสะกด/เขียนไม่เป็นมาตรฐาน** (14 รูปแบบ เหลือ 6 จังหวัดจริง), **อีเมลตัวพิมพ์ใหญ่/มีช่องว่าง**, **quantity ติดลบ**, **customer_id/product_id อ้างอิงไม่พบใน Master Data**, และ **payment ที่ซ้ำ/orphan**

### สรุปคุณภาพข้อมูลหลัง Integration (AFTER) — Data-Quality Funnel

| ขั้นตอน | จำนวนแถว |
|---|---|
| 1. Raw (concat orders ม.ค.+ก.พ.) | 752 |
| 2. ลบ duplicate order_id | 748 |
| 3. ผ่านกติกาทางธุรกิจ (qty/price/discount) | 746 |
| 4. จับคู่ได้กับ Master Data ครบ (customer & product) | 722 |
| 5. ยอดขายที่ใช้ได้จริง (payment_status = PAID) | **660** |

ยอดขายสุทธิรวมทั้งหมด: **10,224,044.08 บาท** (660 ธุรกรรม)

ดูรายละเอียดเชิงลึกทุกรายการที่ถูกคัดออก/แก้ไข พร้อมเหตุผล ได้ในไฟล์ `output/data_quality_report.csv`

---

### คำตอบคำถามวิเคราะห์

**1) หลังรวมไฟล์ orders มีจำนวนแถวเท่าใด และเหลือกี่แถวหลังลบ duplicate?**

หลัง `pd.concat()` ไฟล์ orders เดือนมกราคม (361 แถว) และกุมภาพันธ์ (391 แถว) จะได้ **752 แถว**
หลังลบ `order_id` ที่ซ้ำ (พบ 4 แถวที่เป็น order_id ซ้ำกัน — เก็บแถวล่าสุดตามกติกา) เหลือ **748 แถว**

**2) มีแถวที่ customer_id หรือ product_id ไม่พบใน Master Data อย่างละกี่แถว?**

จากคำสั่งซื้อที่ผ่านกติกาทางธุรกิจแล้ว (746 แถว): `customer_id` ไม่พบใน `customers_crm` จำนวน **22 แถว** และ `product_id` ไม่พบใน `product_master` จำนวน **2 แถว** (ไม่มีแถวที่ไม่พบทั้งสองอย่างพร้อมกัน รวมแถวที่มีปัญหาการอ้างอิง 24 แถว จาก 746 แถว)

**3) มียอดขายที่ใช้ได้จริงกี่ธุรกรรม และยอดขายสุทธิรวมเท่าใด?**

หลังกรองด้วยกติกาทั้งหมด (order ถูกต้อง + จับคู่ customer/product ได้ + payment_status = PAID) เหลือ **660 ธุรกรรม** ยอดขายสุทธิรวม **10,224,044.08 บาท**

**4) จังหวัดใดมียอดขายสุทธิสูงสุด?**

**กรุงเทพมหานคร** มียอดขายสุทธิสูงสุดที่ **2,612,955.87 บาท** จาก 154 ธุรกรรม (ดูรายละเอียดทุกจังหวัดใน `output/summary_by_province.csv`)

| อันดับ | จังหวัด | ธุรกรรม | ยอดขายสุทธิ (บาท) |
|---|---|---|---|
| 1 | กรุงเทพมหานคร | 154 | 2,612,955.87 |
| 2 | ขอนแก่น | 110 | 2,031,943.39 |
| 3 | ระยอง | 120 | 1,523,168.62 |
| 4 | เชียงใหม่ | 104 | 1,477,338.01 |
| 5 | ภูเก็ต | 86 | 1,427,388.74 |
| 6 | ชลบุรี | 86 | 1,151,249.47 |

**5) หมวดสินค้าใดมียอดขายสุทธิสูงสุด?**

**Smartphone** มียอดขายสุทธิสูงสุดที่ **3,092,117.34 บาท** จาก 178 ธุรกรรม (ดูรายละเอียดใน `output/summary_by_category.csv`)

| อันดับ | หมวดสินค้า | ธุรกรรม | ยอดขายสุทธิ (บาท) |
|---|---|---|---|
| 1 | Smartphone | 178 | 3,092,117.34 |
| 2 | Accessory | 180 | 2,710,582.78 |
| 3 | Notebook | 161 | 2,221,495.49 |
| 4 | Smart Home | 141 | 2,199,848.47 |

**6) หากสลับลำดับ merge ก่อน cleaning ผลลัพธ์หรือความเชื่อมั่นของข้อมูลเปลี่ยนอย่างไร?**

หากทำ `pd.merge()` กับ customer/product/payment master **ก่อน** ทำความสะอาดข้อมูล จะเกิดปัญหาความน่าเชื่อถือของข้อมูลหลายจุด:

- **Key ไม่ตรงกันเพราะยังไม่ standardize** — เช่น `customer_id`/`product_id` ที่ยังมีช่องว่างหรือรูปแบบไม่ตรง (จากไฟล์ ก.พ. ที่ชื่อคอลัมน์/รูปแบบต่างจาก ม.ค.) จะทำให้ merge จับคู่ไม่ได้ทั้งที่จริงๆ ควรจับคู่ได้ (false unmatched) — เข้าใจผิดว่าข้อมูลอ้างอิงไม่พบ ทั้งที่จริงเป็นปัญหาการฟอร์แมต
- **แถวซ้ำที่ยังไม่ถูกลบ (order_id ซ้ำ, payment ซ้ำ)** จะถูก merge คูณเป็นหลายแถว (fan-out) ทำให้ `net_sales` และค่าสถิติสรุปถูกนับซ้ำ (over-counting) สูงกว่าความเป็นจริง
- **discount ของเดือน ก.พ. ที่ยังเป็น string ("5%")** จะไม่สามารถนำไปคำนวณ `net_sales` ได้ทันที หรือหากคำนวณผิดพลาดจะได้ตัวเลขที่ผิด (เช่น 5 แทนที่จะเป็น 0.05) ทำให้ยอดขายสุทธิผิดเพี้ยนมหาศาล
- **quantity ติดลบ หรือ unit_price ว่าง** ที่ยังไม่ถูกกรองออกจะปนเข้าไปใน fact table และทำให้ยอดขายสุทธิรวมคลาดเคลื่อน (ลดหรือเพิ่มผิดจากความเป็นจริง)
- โดยสรุป การ **cleaning ก่อน merge เสมอ** ทำให้ตรวจสอบ cardinality (`validate=`) และแถวที่จับคู่ไม่ได้ (`indicator=True`) ได้อย่างแม่นยำ เพราะความคลาดเคลื่อนที่เจอในขั้น merge จะสะท้อนปัญหาการอ้างอิงข้อมูลจริง (referential integrity) ไม่ใช่ปัญหาความสกปรกของ key ที่ควรถูกแก้ไขไปแล้วตั้งแต่ต้น — นี่คือเหตุผลที่ pipeline นี้ยึดลำดับ **Extract → Schema align → Clean/Standardize → Merge/Validate → Load** ตามหลัก ETL ที่ถูกต้อง
